# FireSight-IR | Module 3a — Multi-Branch PINN Training (Final)

**Project:** FireSight-IR — FireSat Protoflight-aligned wildfire detection pipeline  
**Author:** Emmanuel Ibekwe | github.com/Ibekwemmanuel7  
**Platform:** Google Colab free tier (T4 GPU) — Runtime → Change runtime type → T4 GPU

---

## Dataset state going into training

| Item | Value |
|---|---|
| Total patches | 1,149,722 |
| Non-fire (label 0) | 55,460 (4.82%) |
| Wildfire (label 1) | 1,052,247 (91.52%) |
| False-alarm (label 2) | 42,015 (3.65%) |
| Real features | 30 / 36 |
| Land cover | MODIS MCD12Q1 v6.1 ✓ |
| Scalers | feature_scalers_v2.json |
| Class weights | [6.9, 0.36, 9.81] |
| Train / Val / Test | 858,910 / 76,084 / 214,728 |

## Architecture summary

```
CNN branch    (BT_I4, BT_I5, BTD, fire_mask patches 32x32)  → 128-dim
MLP-atm       (ERA5 atmospheric state — 16 features)         →  32-dim
MLP-srf       (MODIS LC + OSM infrastructure — 20 features)  →  32-dim
MLP-derived   (Physics features — 6 features)                →  16-dim
                                                             ──────────
Fusion MLP    208 → 128 → 64
Classification head   64 → 3    non-fire / wildfire / false-alarm
Physics head          64 → 1    transmittance (Beer-Lambert constraint)
```

## Physics loss

```
L = CE(weighted) + 0.1*Beer-Lambert + 0.05*Dynamic-Range + 0.05*Thermal-Realism
```

## Session plan for Colab free tier

| Session | What to run | Expected time |
|---|---|---|
| Session 1 | Sections 0-7 (pre-cache + epochs 1-18) | ~4 hrs |
| Session 2 | Sections 0-7 (auto-resume epochs 19-30) | ~2.5 hrs |
| Final | Sections 8-11 (evaluation + figures) | ~30 min |

**If Colab disconnects at any point:** remount Drive, re-run Sections 0-6, then re-run Section 7. It picks up from the last saved epoch automatically.

---
## Section 0 — Environment
> **GPU required.** If the cell prints CPU, go to Runtime → Change runtime type → T4 GPU first.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted.')

In [ ]:
!pip install torch tqdm scikit-learn matplotlib numpy pandas h5py pyarrow scipy -q

import os, json, time, warnings, sys
import numpy as np
import pandas as pd
import h5py
import scipy.spatial
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import OneCycleLR
from torch.cuda.amp import GradScaler, autocast
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve
from tqdm.auto import tqdm
warnings.filterwarnings('ignore')

device  = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
HAS_AMP = device.type == 'cuda'

print(f'Device : {device}')
if HAS_AMP:
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    print(f'VRAM   : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
    print(f'AMP    : enabled (FP16 — ~1.8x faster than FP32)')
else:
    print('[STOP] No GPU. Go to Runtime → Change runtime type → T4 GPU.')

print(f'PyTorch: {torch.__version__}')

---
## Section 1 — Configuration

In [ ]:
BASE_DIR     = '/content/drive/MyDrive/firesight-ir'
ARCHIVE_PATH = f'{BASE_DIR}/data/processed/patches/firesight_patches.h5'
SPLIT_DIR    = f'{BASE_DIR}/data/splits'
MODEL_DIR    = f'{BASE_DIR}/models'
FIGURES_DIR  = f'{BASE_DIR}/figures'
CACHE_DIR    = f'{BASE_DIR}/data/cache'

# Use v2 weights from Module 2 v2; fall back to v1 if not found
WEIGHTS_PATH = f'{BASE_DIR}/data/scalers/class_weights_v2.json'
if not os.path.exists(WEIGHTS_PATH):
    WEIGHTS_PATH = f'{BASE_DIR}/data/scalers/class_weights.json'
    print('[WARN] Using v1 class weights. Run Module 2 v2 for updated weights.')

for d in [MODEL_DIR, FIGURES_DIR, CACHE_DIR]:
    os.makedirs(d, exist_ok=True)

# Feature dimensions — must match Module 2 output
N_ATM=16; N_SRF=20; N_DERIVED=6; PATCH_SIZE=32; N_CHANNELS=4; N_CLASSES=3

# Training
BATCH_SIZE=1024; N_EPOCHS=30; LR_MAX=3e-4; WEIGHT_DECAY=1e-4; DROPOUT=0.3
NUM_WORKERS=2   # set to 0 if you see DataLoader multiprocessing errors

# Physics loss
LAMBDA_BL=0.10; LAMBDA_DR=0.05; LAMBDA_TH=0.05
BT_I4_FIRE_MIN=310.0; BTD_FIRE_MIN=10.0

# Checkpoint paths
BEST_PATH  = f'{MODEL_DIR}/firesight_pinn_best.pt'
FINAL_PATH = f'{MODEL_DIR}/firesight_pinn_final.pt'
LOG_PATH   = f'{MODEL_DIR}/training_log.json'

SEED=42
torch.manual_seed(SEED); np.random.seed(SEED)
if HAS_AMP: torch.cuda.manual_seed(SEED)

print('Configuration set.')
print(f'  Weights  : {WEIGHTS_PATH}')
print(f'  Batch    : {BATCH_SIZE}  Epochs: {N_EPOCHS}')
print(f'  Physics λ: BL={LAMBDA_BL} DR={LAMBDA_DR} TH={LAMBDA_TH}')

---
## Section 2 — Pre-cache (runs once, auto-skips if up to date)

In [ ]:
CACHE_FILES = {
    'patches': f'{CACHE_DIR}/patches.npy',
    'atm'    : f'{CACHE_DIR}/atm.npy',
    'srf'    : f'{CACHE_DIR}/srf.npy',
    'derived': f'{CACHE_DIR}/derived.npy',
    'labels' : f'{CACHE_DIR}/labels.npy',
    'aux'    : f'{CACHE_DIR}/aux.npy',
}

def build_cache(archive_path, cache_files, chunk=4096):
    print('Building numpy cache (sequential HDF5 read — runs once)...')
    t0 = time.time()
    with h5py.File(archive_path, 'r') as hf:
        N = hf['labels'].shape[0]
        has_derived = 'mlp_derived' in hf
        print(f'  {N:,} samples | mlp_derived: {has_derived}')
        mm = {
            'patches': np.lib.format.open_memmap(cache_files['patches'],'w+','float32',(N,4,32,32)),
            'atm'    : np.lib.format.open_memmap(cache_files['atm'],    'w+','float32',(N,N_ATM)),
            'srf'    : np.lib.format.open_memmap(cache_files['srf'],    'w+','float32',(N,N_SRF)),
            'derived': np.lib.format.open_memmap(cache_files['derived'],'w+','float32',(N,N_DERIVED)),
            'labels' : np.lib.format.open_memmap(cache_files['labels'], 'w+','uint8',  (N,)),
            'aux'    : np.lib.format.open_memmap(cache_files['aux'],    'w+','float32',(N,3)),
        }
        bar = tqdm(range(0,N,chunk), desc='Pre-caching', unit='chunk')
        for start in bar:
            end = min(start+chunk, N); sl = slice(start,end)
            bt4 = hf['cnn/bt_i4'][sl].astype(np.float32)
            bt5 = hf['cnn/bt_i5'][sl].astype(np.float32)
            btd = hf['cnn/bt_diff'][sl].astype(np.float32)
            fm  = hf['cnn/fire_mask'][sl].astype(np.float32)
            mm['patches'][sl] = np.stack([
                (bt4-300.)/50., (bt5-290.)/20., btd/40., fm/9.
            ], axis=1)
            atm = np.nan_to_num(hf['mlp_atm'][sl].astype(np.float32), nan=0.)
            srf = np.nan_to_num(hf['mlp_srf'][sl].astype(np.float32), nan=0.)
            der = np.nan_to_num(
                hf['mlp_derived'][sl].astype(np.float32) if has_derived
                else np.zeros((end-start,N_DERIVED),np.float32), nan=0.)
            mm['atm'][sl]=atm; mm['srf'][sl]=srf; mm['derived'][sl]=der
            mm['labels'][sl]=hf['labels'][sl]
            mm['aux'][sl] = np.stack([
                bt4[:,16,16], btd[:,16,16],
                atm[:,14] if atm.shape[1]>14 else np.full(end-start,0.72)
            ], axis=1)
        for v in mm.values(): v.flush()
    elapsed = time.time()-t0
    print(f'\n✓ Cache built in {elapsed/60:.1f} min')


def is_cache_stale(cache_files, archive_path):
    if not all(os.path.exists(p) for p in cache_files.values()):
        return True, 'missing cache files'
    try:
        cached = np.load(cache_files['labels'], mmap_mode='r')
        with h5py.File(archive_path, 'r') as hf:
            arch = hf['labels'][:]
        idx = np.random.choice(len(arch), min(10000,len(arch)), replace=False)
        if not np.array_equal(cached[idx], arch[idx]):
            return True, 'labels changed since last cache'
    except Exception as e:
        return True, str(e)
    return False, 'ok'


stale, reason = is_cache_stale(CACHE_FILES, ARCHIVE_PATH)
if stale:
    print(f'Cache rebuild needed: {reason}')
    for p in CACHE_FILES.values():
        if os.path.exists(p): os.remove(p)
    build_cache(ARCHIVE_PATH, CACHE_FILES)
else:
    cl = np.load(CACHE_FILES['labels'], mmap_mode='r')
    lc = np.bincount(cl, minlength=3)
    print(f'✓ Cache valid — {len(cl):,} samples')
    print(f'  non-fire={lc[0]:,} | wildfire={lc[1]:,} | false-alarm={lc[2]:,}')

---
## Section 3 — Dataset

In [ ]:
class FireSightDataset(Dataset):
    def __init__(self, cache_files, indices, augment=False):
        self.indices=np.sort(indices); self.augment=augment
        self.patches=np.load(cache_files['patches'],mmap_mode='r')
        self.atm    =np.load(cache_files['atm'],    mmap_mode='r')
        self.srf    =np.load(cache_files['srf'],    mmap_mode='r')
        self.derived=np.load(cache_files['derived'],mmap_mode='r')
        self.labels =np.load(cache_files['labels'], mmap_mode='r')
        self.aux    =np.load(cache_files['aux'],    mmap_mode='r')

    def __len__(self): return len(self.indices)

    def __getitem__(self, i):
        idx=int(self.indices[i]); patch=self.patches[idx].copy()
        if self.augment:
            k=np.random.randint(0,4)
            if k: patch=np.rot90(patch,k,axes=(1,2)).copy()
            if np.random.rand()>0.5: patch=np.flip(patch,axis=2).copy()
        return (
            torch.from_numpy(patch),
            torch.from_numpy(self.atm[idx].copy()),
            torch.from_numpy(self.srf[idx].copy()),
            torch.from_numpy(self.derived[idx].copy()),
            torch.tensor(int(self.labels[idx]),dtype=torch.long),
            torch.from_numpy(self.aux[idx].copy()),
        )

print('FireSightDataset defined.')

---
## Section 4 — Model

In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self,in_dim,out_dim,dropout=0.2):
        super().__init__()
        self.net=nn.Sequential(
            nn.Linear(in_dim,out_dim),nn.BatchNorm1d(out_dim),nn.ReLU(),
            nn.Dropout(dropout),nn.Linear(out_dim,out_dim),nn.BatchNorm1d(out_dim))
        self.proj=nn.Linear(in_dim,out_dim) if in_dim!=out_dim else nn.Identity()
    def forward(self,x): return F.relu(self.net(x)+self.proj(x))

class CNNBranch(nn.Module):
    def __init__(self,in_ch=4,dropout=0.2):
        super().__init__()
        self.enc=nn.Sequential(
            nn.Conv2d(in_ch,32,3,padding=1),nn.BatchNorm2d(32),nn.ReLU(True),
            nn.Conv2d(32,32,3,padding=1),nn.BatchNorm2d(32),nn.ReLU(True),
            nn.MaxPool2d(2),nn.Dropout2d(0.1),
            nn.Conv2d(32,64,3,padding=1),nn.BatchNorm2d(64),nn.ReLU(True),
            nn.Conv2d(64,64,3,padding=1),nn.BatchNorm2d(64),nn.ReLU(True),
            nn.MaxPool2d(2),nn.Dropout2d(0.1),
            nn.Conv2d(64,128,3,padding=1),nn.BatchNorm2d(128),nn.ReLU(True),
            nn.MaxPool2d(2))
        self.gap=nn.AdaptiveAvgPool2d(1); self.drop=nn.Dropout(dropout)
    def forward(self,x): return self.drop(self.gap(self.enc(x)).flatten(1))

class FireSightPINN(nn.Module):
    """
    Multi-branch PINN: CNN(128)+MLP-atm(32)+MLP-srf(32)+MLP-der(16)=208 → 3-class
    + transmittance head for Beer-Lambert physics loss.
    """
    def __init__(self,n_atm=16,n_srf=20,n_der=6,n_cls=3,drop=0.3):
        super().__init__()
        self.cnn=CNNBranch(4,drop)
        self.atm=nn.Sequential(ResidualBlock(n_atm,64),ResidualBlock(64,32))
        self.srf=nn.Sequential(ResidualBlock(n_srf,64),ResidualBlock(64,32))
        self.der=nn.Sequential(
            nn.Linear(n_der,32),nn.BatchNorm1d(32),nn.ReLU(),nn.Dropout(0.1),
            nn.Linear(32,16),nn.BatchNorm1d(16),nn.ReLU())
        self.fusion=nn.Sequential(
            nn.Linear(208,128),nn.BatchNorm1d(128),nn.ReLU(),nn.Dropout(drop),
            nn.Linear(128,64),nn.BatchNorm1d(64),nn.ReLU(),nn.Dropout(drop))
        self.cls  =nn.Linear(64,n_cls)
        self.trans=nn.Sequential(nn.Linear(64,16),nn.ReLU(),nn.Linear(16,1),nn.Sigmoid())

    def forward(self,patch,atm,srf,der):
        feat=self.fusion(torch.cat([self.cnn(patch),self.atm(atm),self.srf(srf),self.der(der)],dim=1))
        return self.cls(feat),self.trans(feat)

_m=FireSightPINN(N_ATM,N_SRF,N_DERIVED,N_CLASSES,DROPOUT).to(device)
n_params=sum(p.numel() for p in _m.parameters())
print(f'FireSightPINN: {n_params:,} parameters')
with torch.no_grad():
    _l,_t=_m(torch.randn(4,N_CHANNELS,PATCH_SIZE,PATCH_SIZE).to(device),
              torch.randn(4,N_ATM).to(device),torch.randn(4,N_SRF).to(device),
              torch.randn(4,N_DERIVED).to(device))
print(f'Dry run: logits={_l.shape}, transmittance={_t.shape} ✓'); del _m

---
## Section 5 — Physics loss

In [ ]:
class PINNLoss(nn.Module):
    def __init__(self,weights,lbl=0.1,ldr=0.05,lth=0.05,bt_min=310.,btd_min=10.,device='cpu'):
        super().__init__()
        self.lbl,self.ldr,self.lth=lbl,ldr,lth
        self.bt_min,self.btd_min=bt_min,btd_min
        w=torch.tensor(weights,dtype=torch.float32).to(device)
        self.ce=nn.CrossEntropyLoss(weight=w); self.mse=nn.MSELoss()
    def forward(self,logits,trans,labels,aux):
        L_ce=self.ce(logits,labels)
        L_bl=self.mse(trans,aux[:,2:3])
        p_wf=F.softmax(logits,dim=1)[:,1]
        L_dr=(p_wf*(aux[:,0]<self.bt_min).float()).mean()
        L_th=(p_wf*(aux[:,1]<self.btd_min).float()).mean()
        total=L_ce+self.lbl*L_bl+self.ldr*L_dr+self.lth*L_th
        return total,{'ce':L_ce.item(),'bl':L_bl.item(),'dr':L_dr.item(),'th':L_th.item()}

print('PINNLoss defined.')

---
## Section 6 — Splits and DataLoaders

In [ ]:
train_idx=np.load(f'{SPLIT_DIR}/train_index.npy')
val_idx  =np.load(f'{SPLIT_DIR}/val_index.npy')
test_idx =np.load(f'{SPLIT_DIR}/test_index.npy')
print(f'Splits — Train:{len(train_idx):,} Val:{len(val_idx):,} Test:{len(test_idx):,}')

with open(WEIGHTS_PATH) as f: cw=json.load(f)
if isinstance(cw,list): class_weights=cw
elif isinstance(cw,dict):
    class_weights=cw.get('_list',[float(cw.get(str(i),[6.9,0.36,9.81][i])) for i in range(3)])
else: class_weights=[6.9,0.36,9.81]
print(f'Class weights: {[round(w,3) for w in class_weights]}')

# Label check
cl=np.load(CACHE_FILES['labels'],mmap_mode='r')
tl=cl[train_idx]; lc=np.bincount(tl,minlength=3)
print(f'Train labels: nf={lc[0]:,} wf={lc[1]:,} fa={lc[2]:,}')
if lc[0]<100: print('[WARN] Very few non-fire in training set — check cache rebuild')

pin=device.type=='cuda'
lkw=dict(num_workers=NUM_WORKERS,pin_memory=pin,
         prefetch_factor=2 if NUM_WORKERS>0 else None,persistent_workers=False)

train_ds=FireSightDataset(CACHE_FILES,train_idx,augment=True)
val_ds  =FireSightDataset(CACHE_FILES,val_idx,  augment=False)
test_ds =FireSightDataset(CACHE_FILES,test_idx, augment=False)

train_loader=DataLoader(train_ds,batch_size=BATCH_SIZE,  shuffle=True, **lkw)
val_loader  =DataLoader(val_ds,  batch_size=BATCH_SIZE*2,shuffle=False,**lkw)
test_loader =DataLoader(test_ds, batch_size=BATCH_SIZE*2,shuffle=False,**lkw)

print(f'Train batches: {len(train_loader):,}')
t0=time.time(); _=next(iter(train_loader)); bt=time.time()-t0
print(f'First batch: {bt:.2f}s  (target <0.5s)')
if bt>2: print('[WARN] Slow batch. Try NUM_WORKERS=0.')
del _

---
## Section 7 — Training

> **Just re-run this cell each session. It automatically resumes from the last saved checkpoint.**
>
> Progress bar shows: `loss`, `acc`, and `lr` live per batch.  
> After each epoch: validation loss, per-class accuracy, and physics loss breakdown.  
> A `✓ Best model saved` line appears whenever a new best is found.

In [ ]:
# ── Init model, loss, scaler ──────────────────────────────────────────────────
model    =FireSightPINN(N_ATM,N_SRF,N_DERIVED,N_CLASSES,DROPOUT).to(device)
criterion=PINNLoss(class_weights,LAMBDA_BL,LAMBDA_DR,LAMBDA_TH,
                   BT_I4_FIRE_MIN,BTD_FIRE_MIN,device)
scaler   =GradScaler(enabled=HAS_AMP)

best_val_loss=float('inf'); training_log=[]; start_epoch=0

# ── Resume ────────────────────────────────────────────────────────────────────
if os.path.exists(BEST_PATH):
    ckpt=torch.load(BEST_PATH,map_location=device)
    model.load_state_dict(ckpt['model_state_dict'])
    best_val_loss=ckpt.get('val_loss',float('inf'))
    start_epoch  =ckpt.get('epoch',0)
    if os.path.exists(LOG_PATH):
        with open(LOG_PATH) as f: training_log=json.load(f)
    print(f'✓ Resumed from epoch {start_epoch} | best val_loss={best_val_loss:.4f}')
    print(f'  Epochs remaining: {N_EPOCHS-start_epoch}')
else:
    print('Starting fresh training')

if start_epoch>=N_EPOCHS:
    print(f'\n✓ Training complete ({N_EPOCHS} epochs). Proceed to Section 8.')
else:
    optimizer=AdamW(model.parameters(),lr=LR_MAX,weight_decay=WEIGHT_DECAY)
    if os.path.exists(BEST_PATH):
        ckpt=torch.load(BEST_PATH,map_location=device)
        if 'optimizer_state' in ckpt:
            optimizer.load_state_dict(ckpt['optimizer_state'])

    completed_steps=start_epoch*len(train_loader)
    scheduler=OneCycleLR(optimizer,max_lr=LR_MAX,
                         steps_per_epoch=len(train_loader),epochs=N_EPOCHS,
                         pct_start=0.1,anneal_strategy='cos',
                         last_epoch=completed_steps-1)

    print(f'\nTraining epochs {start_epoch+1}–{N_EPOCHS}')
    print(f'  ~{len(train_loader)*8//60}–{len(train_loader)*10//60} min/epoch on T4 with AMP')
    print(f'  Checkpoint saved every epoch to: {BEST_PATH}\n')

    for epoch in range(start_epoch,N_EPOCHS):
        t0=time.time()

        # ── Train ─────────────────────────────────────────────────────────────
        model.train()
        total_loss=0; comp_acc={'ce':0,'bl':0,'dr':0,'th':0}; correct=n_total=0

        bar=tqdm(train_loader,desc=f'Ep {epoch+1:02d}/{N_EPOCHS}',ncols=110,leave=True)
        for patch,atm,srf,der,labels,aux in bar:
            patch =patch.to(device, non_blocking=True)
            atm   =atm.to(device,   non_blocking=True)
            srf   =srf.to(device,   non_blocking=True)
            der   =der.to(device,   non_blocking=True)
            labels=labels.to(device,non_blocking=True)
            aux   =aux.to(device,   non_blocking=True)

            optimizer.zero_grad()
            with autocast(enabled=HAS_AMP):
                logits,trans=model(patch,atm,srf,der)
                loss,comp   =criterion(logits,trans,labels,aux)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(),1.0)
            scaler.step(optimizer); scaler.update(); scheduler.step()

            bs=len(labels); total_loss+=loss.item()*bs
            for k in comp_acc: comp_acc[k]+=comp[k]*bs
            correct+=(logits.argmax(1)==labels).sum().item(); n_total+=bs
            bar.set_postfix(loss=f'{loss.item():.3f}',
                            acc =f'{correct/n_total:.3f}',
                            lr  =f'{scheduler.get_last_lr()[0]:.1e}')

        train_loss=total_loss/n_total; train_acc=correct/n_total
        avg_comp={k:v/n_total for k,v in comp_acc.items()}

        # ── Validate ──────────────────────────────────────────────────────────
        model.eval(); val_total=0; vp,vl=[],[]
        with torch.no_grad():
            for patch,atm,srf,der,labels,aux in val_loader:
                with autocast(enabled=HAS_AMP):
                    logits,trans=model(patch.to(device),atm.to(device),
                                       srf.to(device),der.to(device))
                    loss,_=criterion(logits,trans,labels.to(device),aux.to(device))
                val_total+=loss.item()*len(labels)
                vp.append(logits.argmax(1).cpu().numpy())
                vl.append(labels.numpy())

        val_loss  =val_total/len(val_idx)
        val_preds =np.concatenate(vp); val_labels=np.concatenate(vl)
        val_acc   =(val_preds==val_labels).mean()
        pca={}
        for cls,nm in [(0,'nf'),(1,'wf'),(2,'fa')]:
            m=val_labels==cls
            pca[nm]=float((val_preds[m]==cls).mean()) if m.sum()>0 else 0.
        elapsed=time.time()-t0

        print(f'  → val={val_loss:.4f} acc={val_acc:.3f} | '
              f'nf={pca["nf"]:.3f} wf={pca["wf"]:.3f} fa={pca["fa"]:.3f} | {elapsed:.0f}s')
        print(f'     CE={avg_comp["ce"]:.4f} BL={avg_comp["bl"]:.4f} '
              f'DR={avg_comp["dr"]:.4f} TH={avg_comp["th"]:.4f}')

        epoch_log={'epoch':epoch+1,'train_loss':round(train_loss,4),
                   'train_acc':round(train_acc,4),'val_loss':round(val_loss,4),
                   'val_acc':round(val_acc,4),'elapsed_s':round(elapsed,1),
                   'loss_components':{k:round(v,5) for k,v in avg_comp.items()},
                   'per_class_val_acc':{k:round(v,4) for k,v in pca.items()}}
        training_log.append(epoch_log)

        if val_loss<best_val_loss:
            best_val_loss=val_loss
            torch.save({'epoch':epoch+1,'model_state_dict':model.state_dict(),
                        'optimizer_state':optimizer.state_dict(),
                        'val_loss':val_loss,'val_acc':val_acc,
                        'class_weights':class_weights,'per_class_acc':pca},BEST_PATH)
            print(f'     ✓ Best model saved (val_loss={val_loss:.4f})')

        with open(LOG_PATH,'w') as f: json.dump(training_log,f,indent=2)

    torch.save({'epoch':N_EPOCHS,'model_state_dict':model.state_dict()},FINAL_PATH)
    print(f'\n✓ Training complete | best val_loss: {best_val_loss:.4f}')

---
## Section 8 — Training curves

In [ ]:
BG,PANEL='#0D1117','#161B22'
TEXT,SUBTEXT,BORDER='#E6EDF3','#8B949E','#30363D'
plt.rcParams.update({'figure.facecolor':BG,'axes.facecolor':PANEL,
                     'axes.edgecolor':BORDER,'text.color':TEXT,
                     'xtick.color':SUBTEXT,'ytick.color':SUBTEXT,
                     'axes.labelcolor':SUBTEXT,'grid.color':BORDER})

with open(LOG_PATH) as f: log=json.load(f)
ep =[e['epoch']      for e in log]
trl=[e['train_loss'] for e in log]; vll=[e['val_loss']  for e in log]
tra=[e['train_acc']  for e in log]; vla=[e['val_acc']   for e in log]
ce_=[e['loss_components']['ce'] for e in log]
bl_=[e['loss_components']['bl'] for e in log]
dr_=[e['loss_components']['dr'] for e in log]
th_=[e['loss_components']['th'] for e in log]
nfa=[e['per_class_val_acc'].get('nf',0) for e in log]
wfa=[e['per_class_val_acc'].get('wf',0) for e in log]
faa=[e['per_class_val_acc'].get('fa',0) for e in log]

fig,axes=plt.subplots(1,3,figsize=(16,5.5),facecolor=BG)
fig.patch.set_facecolor(BG)

ax=axes[0]
ax.plot(ep,trl,color='#4FC3F7',lw=2,label='Train')
ax.plot(ep,vll,color='#E85D04',lw=2,label='Val (2023)')
ax.set_title('Total loss',color=TEXT,fontsize=11); ax.set_xlabel('Epoch')
ax.legend(facecolor=PANEL,edgecolor=BORDER,labelcolor=TEXT,fontsize=9)
ax.grid(alpha=0.2); ax.spines[['top','right']].set_visible(False)

ax=axes[1]
ax.plot(ep,tra,color='#4FC3F7',lw=2,label='Train')
ax.plot(ep,vla,color='#E85D04',lw=2,label='Val overall')
ax.plot(ep,nfa,color='#3A5C8A',lw=1.5,ls='--',label='Val non-fire')
ax.plot(ep,wfa,color='#78A830',lw=1.5,ls='--',label='Val wildfire')
ax.plot(ep,faa,color='#D97706',lw=1.5,ls='--',label='Val false-alarm')
ax.set_ylim(0,1.05); ax.set_title('Accuracy',color=TEXT,fontsize=11); ax.set_xlabel('Epoch')
ax.legend(facecolor=PANEL,edgecolor=BORDER,labelcolor=TEXT,fontsize=8,ncol=2)
ax.grid(alpha=0.2); ax.spines[['top','right']].set_visible(False)

ax=axes[2]
ax.plot(ep,ce_,color='#FFD700',lw=2,label='CE')
ax.plot(ep,bl_,color='#4CAF50',lw=2,label='Beer-Lambert')
ax.plot(ep,dr_,color='#FF69B4',lw=2,label='Dynamic range')
ax.plot(ep,th_,color='#00BCD4',lw=2,label='Thermal realism')
ax.set_title('Physics loss decomposition',color=TEXT,fontsize=11); ax.set_xlabel('Epoch')
ax.legend(facecolor=PANEL,edgecolor=BORDER,labelcolor=TEXT,fontsize=9)
ax.grid(alpha=0.2); ax.spines[['top','right']].set_visible(False)

fig.suptitle('FireSight-IR  |  Module 3a — Training Curves',color=TEXT,fontsize=13,y=1.01)
plt.tight_layout()
out=f'{FIGURES_DIR}/03a_training_curves.png'
plt.savefig(out,dpi=160,bbox_inches='tight',facecolor=BG); plt.show()
print(f'Saved → {out}')

---
## Section 9 — Evaluation

In [ ]:
ckpt=torch.load(BEST_PATH,map_location=device)
model=FireSightPINN(N_ATM,N_SRF,N_DERIVED,N_CLASSES,DROPOUT).to(device)
model.load_state_dict(ckpt['model_state_dict'])
print(f'Best checkpoint: epoch {ckpt["epoch"]} | val_loss={ckpt["val_loss"]:.4f}')

@torch.no_grad()
def evaluate(loader,name):
    model.eval(); preds,lbls,probs=[],[],[]
    for patch,atm,srf,der,labels,aux in loader:
        with autocast(enabled=HAS_AMP):
            logits,_=model(patch.to(device),atm.to(device),srf.to(device),der.to(device))
        preds.append(logits.argmax(1).cpu().numpy())
        lbls.append(labels.numpy())
        probs.append(F.softmax(logits,dim=1).cpu().numpy())
    p,l,pr=np.concatenate(preds),np.concatenate(lbls),np.concatenate(probs)
    print(f'\n═══ {name} ═══')
    print(classification_report(l,p,target_names=['non-fire','wildfire','false-alarm'],digits=4))
    return p,l,pr

test_preds,test_labels,test_probs=evaluate(test_loader,'Test set (2018-2022 held-out 20%)')
val_preds, val_labels, val_probs =evaluate(val_loader, 'Val set (2023 fully held-out)')

In [ ]:
# ── Confusion matrices ─────────────────────────────────────────────────────────
fig,axes=plt.subplots(1,2,figsize=(13,6),facecolor=BG)
fig.patch.set_facecolor(BG)
cnames=['Non-fire','Wildfire','False-alarm']
for ax,(preds,labels,title) in zip(axes,[
    (test_preds,test_labels,'Test (2018-2022)'),
    (val_preds, val_labels, 'Val (2023)'),
]):
    cm=confusion_matrix(labels,preds,normalize='true')
    im=ax.imshow(cm,cmap='Blues',vmin=0,vmax=1)
    ax.set_xticks(range(3)); ax.set_yticks(range(3))
    ax.set_xticklabels(cnames,rotation=30,ha='right',color=TEXT,fontsize=9)
    ax.set_yticklabels(cnames,color=TEXT,fontsize=9)
    ax.set_xlabel('Predicted',color=SUBTEXT); ax.set_ylabel('True',color=SUBTEXT)
    ax.set_title(title,color=TEXT,fontsize=11)
    for i in range(3):
        for j in range(3):
            ax.text(j,i,f'{cm[i,j]:.3f}',ha='center',va='center',
                    color='white' if cm[i,j]<0.5 else 'black',fontsize=10)
    plt.colorbar(im,ax=ax,fraction=0.046)
fig.suptitle('FireSight-IR | Module 3a — Confusion Matrices',color=TEXT,fontsize=12,y=1.01)
plt.tight_layout()
out=f'{FIGURES_DIR}/03a_confusion_matrix.png'
plt.savefig(out,dpi=160,bbox_inches='tight',facecolor=BG); plt.show()
print(f'Saved → {out}')

In [ ]:
# ── ROC curves ─────────────────────────────────────────────────────────────────
fig,axes=plt.subplots(1,2,figsize=(13,6),facecolor=BG)
fig.patch.set_facecolor(BG)
roc_colors={'non-fire':'#3A5C8A','wildfire':'#E85D04','false-alarm':'#D97706'}
for ax,(probs,labels,title) in zip(axes,[
    (test_probs,test_labels,'Test (2018-2022)'),
    (val_probs, val_labels, 'Val (2023)'),
]):
    ax.set_facecolor(PANEL)
    ax.plot([0,1],[0,1],color=BORDER,lw=1,ls='--')
    for ci,(cname,col) in enumerate(roc_colors.items()):
        bl=(labels==ci).astype(int)
        if bl.sum()==0: continue
        fpr,tpr,_=roc_curve(bl,probs[:,ci])
        auc=roc_auc_score(bl,probs[:,ci])
        ax.plot(fpr,tpr,color=col,lw=2,label=f'{cname}  AUC={auc:.4f}')
    ax.set_xlabel('FPR',color=SUBTEXT); ax.set_ylabel('TPR',color=SUBTEXT)
    ax.set_title(title,color=TEXT,fontsize=11)
    ax.legend(facecolor=PANEL,edgecolor=BORDER,labelcolor=TEXT,fontsize=9)
    ax.grid(alpha=0.2); ax.spines[['top','right']].set_visible(False)
fig.suptitle('FireSight-IR | Module 3a — ROC Curves',color=TEXT,fontsize=12,y=1.01)
plt.tight_layout()
out=f'{FIGURES_DIR}/03a_roc_curves.png'
plt.savefig(out,dpi=160,bbox_inches='tight',facecolor=BG); plt.show()
print(f'Saved → {out}')

---
## Section 10 — Physics constraint verification

In [ ]:
@torch.no_grad()
def collect_physics(loader,max_n=50000):
    model.eval(); trans_l,aux_l,pred_l,lbl_l=[],[],[],[]; n=0
    for patch,atm,srf,der,labels,aux in loader:
        if n>=max_n: break
        with autocast(enabled=HAS_AMP):
            logits,trans=model(patch.to(device),atm.to(device),srf.to(device),der.to(device))
        trans_l.append(trans.cpu().numpy()); aux_l.append(aux.numpy())
        pred_l.append(logits.argmax(1).cpu().numpy()); lbl_l.append(labels.numpy()); n+=len(labels)
    return (np.concatenate(trans_l)[:max_n],np.concatenate(aux_l)[:max_n],
            np.concatenate(pred_l)[:max_n],np.concatenate(lbl_l)[:max_n])

trans,aux_v,preds_v,lbls_v=collect_physics(val_loader)
bt4v=aux_v[:,0]; btdv=aux_v[:,1]; beerv=aux_v[:,2]; tv=trans.flatten()

fig,axes=plt.subplots(1,3,figsize=(16,5.5),facecolor=BG)
fig.patch.set_facecolor(BG)
cp={0:'#3A5C8A',1:'#E85D04',2:'#D97706'}; np_={0:'non-fire',1:'wildfire',2:'false-alarm'}

ax=axes[0]; ax.set_facecolor(PANEL)
ax.scatter(beerv[:5000],tv[:5000],alpha=0.15,s=1,color='#4CAF50')
ax.plot([0,1],[0,1],color='#E85D04',lw=1.5,ls='--',label='ideal')
corr=np.corrcoef(beerv,tv)[0,1]
ax.set_title(f'Beer-Lambert (r={corr:.3f})',color=TEXT,fontsize=11)
ax.set_xlabel('ERA5 TCWV transmittance',color=SUBTEXT); ax.set_ylabel('Predicted',color=SUBTEXT)
ax.legend(facecolor=PANEL,edgecolor=BORDER,labelcolor=TEXT,fontsize=8)
ax.grid(alpha=0.2); ax.spines[['top','right']].set_visible(False)

for ax,data,threshold,xlabel,title_txt in [
    (axes[1],bt4v,BT_I4_FIRE_MIN,'BT_I4 centre (K)','BT_I4 by predicted class'),
    (axes[2],btdv,BTD_FIRE_MIN,  'BTD centre (K)',   'BTD by predicted class'),
]:
    ax.set_facecolor(PANEL)
    for cls in [0,1,2]:
        v=data[preds_v==cls]
        if len(v)>0: ax.hist(v,bins=60,alpha=0.5,color=cp[cls],density=True,label=np_[cls])
    ax.axvline(threshold,color='white',lw=1.5,ls=':',label=f'threshold={threshold:.0f}K')
    ax.set_title(title_txt,color=TEXT,fontsize=10); ax.set_xlabel(xlabel,color=SUBTEXT)
    ax.legend(facecolor=PANEL,edgecolor=BORDER,labelcolor=TEXT,fontsize=8)
    ax.grid(alpha=0.2); ax.spines[['top','right']].set_visible(False)

fig.suptitle('FireSight-IR | Module 3a — Physics Constraint Analysis (Val 2023)',
             color=TEXT,fontsize=12,y=1.01)
plt.tight_layout()
out=f'{FIGURES_DIR}/03a_physics_analysis.png'
plt.savefig(out,dpi=160,bbox_inches='tight',facecolor=BG); plt.show()
print(f'Saved → {out}')

---
## Section 11 — Summary

In [ ]:
print('='*60)
print('  FireSight-IR  |  Module 3a Complete')
print('='*60)
print(f'  Parameters   : {sum(p.numel() for p in model.parameters()):,}')
print(f'  Epochs       : {len(training_log)}')
print(f'  Best val loss: {best_val_loss:.4f}')
best_ep=min(training_log,key=lambda e:e['val_loss'])
print(f'  Best epoch   : {best_ep["epoch"]} (val_acc={best_ep["val_acc"]:.4f})')
print(f'  Per-class    : {best_ep["per_class_val_acc"]}')
print()
for path,desc in [
    (BEST_PATH,'Best model weights'),
    (FINAL_PATH,'Final epoch weights'),
    (LOG_PATH,'Training log'),
    (f'{FIGURES_DIR}/03a_training_curves.png','Training curves'),
    (f'{FIGURES_DIR}/03a_confusion_matrix.png','Confusion matrices'),
    (f'{FIGURES_DIR}/03a_roc_curves.png','ROC curves'),
    (f'{FIGURES_DIR}/03a_physics_analysis.png','Physics analysis'),
]:
    e='✓' if os.path.exists(path) else '✗'
    print(f'  {e}  {desc}')
print()
print('  Paste your classification report here after training completes.')
print('  Next: Module 3b — False-alarm discriminator')
print('='*60)